# ABR raw-feature univariate EDA (vs. synapse count)

**Run** from repository root (folder containing `utils/`).

- **10 raw features** (pre-`log1p` / pre-scaling) vs. **synapses per IHC**
- **Dual grain:** `long` (all SPL levels) and `animal_freq` (80 dB nearest per animal×frequency)
- **Cohorts:** A (Liberman) | B (Brad); points colored/marked by true noise group

**Print-ready manuscript figures (animal × frequency @ 80 dB):**

- **Figure 4:** `figures/eda/figure_04_raw_vs_synapses_cohort_A.{png,svg}` + `figure_04_caption.txt`
- **Figure 5:** `figures/eda/figure_05_raw_vs_synapses_cohort_B.{png,svg}` + `figure_05_caption.txt`
- **Combined 5×4:** `figures/eda/figure_combined_raw_vs_synapses_cohorts_5x4.{png,svg}` + `figure_combined_caption.txt`
- Spearman table: `figures/cache/eda/figure_04_05_spearman_animal_freq.csv`

**Exploratory (tall layout):** `figures/eda/abr_raw_vs_synapses_scatter_long.{png,svg}` — all SPL levels, 10×2 combined cohorts.

In [1]:
%matplotlib inline

import logging
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO)

_CWD = Path.cwd().resolve()
if not (_CWD / "utils" / "abr_univariate_eda.py").is_file():
    raise RuntimeError(
        f"Wrong cwd: {_CWD}. Open the repo root (contains utils/abr_univariate_eda.py)."
    )

from utils.abr_univariate_eda import (
    CROSS_COHORT_NOTE,
    GRAIN_ANIMAL_FREQ,
    GRAIN_LONG,
    spearman_table_wide,
    verify_feature_identity,
    run_univariate_eda,
)
from utils.nn_stage2_data import load_nn_stage2_data

verify_feature_identity()
data = load_nn_stage2_data()
result = run_univariate_eda(data)
print("Report:", result["report_path"])
print("Skew table:", result["skew_path"])
if result.get("figure_04_path"):
    print("Figure 4:", result["figure_04_path"])
    print("Figure 5:", result["figure_05_path"])
    if result.get("figure_combined_path"):
        print("Combined 5×4:", result["figure_combined_path"])
    print(
        "Captions:",
        Path("figures/eda/figure_04_caption.txt"),
        Path("figures/eda/figure_05_caption.txt"),
        Path("figures/eda/figure_combined_caption.txt"),
    )
if result.get("figure_04_05_spearman_path"):
    print("Spearman table:", result["figure_04_05_spearman_path"])
display(result["skew_tbl"].sort_values(["cohort", "feature"]))

INFO:utils.abr_univariate_eda:EDA features: 10 columns; distance=peak-to-trough latency; slope=-amplitude/distance (Wave I); log1p candidates=['PeakICentralCurvature', 'PeakIEarlyCurvature', 'PeakILateCurvature', 'TroughICentralCurvature', 'TroughIEarlyCurvature', 'TroughILateCurvature', 'total_variance']


INFO:utils.abr_univariate_eda:Excluded from RAW_EDA_FEATURES: ['Slope_all', 'Slope_high4', 'p1_latency']


INFO:utils.abr_univariate_eda:EDA features: 10 columns; distance=peak-to-trough latency; slope=-amplitude/distance (Wave I); log1p candidates=['PeakICentralCurvature', 'PeakIEarlyCurvature', 'PeakILateCurvature', 'TroughICentralCurvature', 'TroughIEarlyCurvature', 'TroughILateCurvature', 'total_variance']


INFO:utils.abr_univariate_eda:Excluded from RAW_EDA_FEATURES: ['Slope_all', 'Slope_high4', 'p1_latency']


Report: figures/cache/eda/abr_univariate_eda_report.md


## Feature identity

- `distance` = **peak-to-trough latency** (trough time − peak time)
- `slope` = **Wave I slope** (−amplitude / distance); not `Slope_all` / `Slope_high4`

## Cross-cohort comparability

See generated report; summary:

> Cross-cohort note (also in `abr_univariate_eda_report.md`)

In [2]:
print(CROSS_COHORT_NOTE)

Brad vs. Liberman **amplitude** and **peak-to-trough latency** (`distance`) are constructed with the **same formulas** but on **different waveform grids**. Cohort B waveforms were resampled to the Liberman timebase (**LibT**), so numeric ranges are **broadly comparable** across cohorts, with small residual differences from **resampling interpolation** and **landmark detection method** (in-house peaks/troughs vs IO latencies). Cross-cohort feature comparisons in the EDA should be interpreted with this in mind, but are **not precluded**.


## Spearman tables

In [3]:
print("### Long grain (all SPL levels)")
display(spearman_table_wide(result["long_tbl"]))
print("\n### Animal × frequency @ 80 dB nearest")
display(spearman_table_wide(result["af_tbl"]))

### Long grain (all SPL levels)


stratum,A / all,A / higher,A / lower,B / all,B / higher,B / lower
feature_label,,,,,,
Amplitude,0.085344,0.132076,0.023347,0.465105,0.486210,0.396091
Peak I curvature (central),0.010232,0.009212,0.019021,0.001872,0.051417,-0.024043
Peak I curvature (early),-0.042514,-0.059009,-0.026273,-0.250654,-0.314727,-0.209227
Peak I curvature (late),-0.076838,-0.100527,-0.038039,-0.317905,-0.400383,-0.245754
Peak-to-trough latency,-0.046552,-0.011738,-0.048031,0.176211,0.127019,0.127824
Total variance,0.074804,0.125108,0.017986,0.465099,0.484055,0.391358
Trough I curvature (central),-0.002476,0.007137,-0.027503,0.057562,0.106966,0.028420
Trough I curvature (early),-0.061804,-0.068211,-0.020869,-0.243702,-0.354778,-0.171016
Trough I curvature (late),-0.046120,-0.072384,-0.027721,-0.287410,-0.345317,-0.248404



### Animal × frequency @ 80 dB nearest


stratum,A / all,A / higher,A / lower,B / all,B / higher,B / lower
feature_label,,,,,,
Amplitude,0.271815,0.341599,0.099329,0.630969,0.637868,0.588076
Peak I curvature (central),0.052174,0.056003,-0.037484,-0.104485,-0.046806,-0.142640
Peak I curvature (early),-0.214622,-0.257883,-0.118178,-0.456171,-0.544756,-0.432035
Peak I curvature (late),-0.243644,-0.297916,-0.072535,-0.481758,-0.523722,-0.482398
Peak-to-trough latency,0.044642,0.082339,-0.031439,0.200850,0.204927,0.033570
Total variance,0.266018,0.335783,0.087652,0.632223,0.641447,0.568566
Trough I curvature (central),0.028870,0.072052,-0.012640,0.069936,0.115260,0.039301
Trough I curvature (early),-0.207761,-0.235579,-0.064078,-0.388909,-0.512124,-0.328859
Trough I curvature (late),-0.092060,-0.195281,0.005214,-0.348312,-0.470889,-0.326660


## Grain comparison (Δρ = ρ_long − ρ_animal_freq)

In [4]:
cmp = result["grain_cmp"].sort_values("abs_delta_rho", ascending=False)
display(cmp.head(20))

,feature,cohort,noise_group,rho_long,n_long,p_long,rho_animal_freq,n_animal_freq,p_animal_freq,delta_rho,abs_delta_rho
17,PeakILateCurvature,B,lower,-0.245754,2038,2.051958e-29,-0.482398,199,5.401143e-13,0.236644,0.236644
10,PeakIEarlyCurvature,B,higher,-0.314727,1516,3.300466e-36,-0.544756,161,7.997679e-14,0.230029,0.230029
49,slope,A,higher,-0.126038,5575,3.510482e-21,-0.353587,323,6.043931e-11,0.227549,0.227549
11,PeakIEarlyCurvature,B,lower,-0.209227,2038,1.355208e-21,-0.432035,199,1.869323e-10,0.222808,0.222808
53,slope,B,lower,-0.376802,2038,9.345074e-70,-0.592429,199,3.058386e-20,0.215627,0.215627
55,total_variance,A,higher,0.125108,5575,6.847128e-21,0.335783,323,5.938010e-10,-0.210675,0.210675
37,amplitude,A,higher,0.132076,5575,4.047828e-23,0.341599,323,2.859833e-10,-0.209523,0.209523
9,PeakIEarlyCurvature,B,all,-0.250654,3554,4.758011e-52,-0.456171,360,6.666393e-20,0.205517,0.205517
48,slope,A,all,-0.091374,12187,5.127494e-24,-0.290502,616,1.914766e-13,0.199128,0.199128
7,PeakIEarlyCurvature,A,higher,-0.059009,5575,1.039543e-05,-0.257883,323,2.647392e-06,0.198875,0.198875


## Generated report

In [5]:
from IPython.display import Markdown

Markdown(result["report_path"].read_text())

# ABR raw-feature univariate EDA report

## 1. Feature identity
- `distance` = peak-to-trough latency (trough time − peak time).
- `slope` = Wave I slope (−amplitude / distance).
- Excluded: `Slope_all`, `Slope_high4` (Buran amplitude–SPL slopes).

## 2. Cross-cohort note
Brad vs. Liberman **amplitude** and **peak-to-trough latency** (`distance`) are constructed with the **same formulas** but on **different waveform grids**. Cohort B waveforms were resampled to the Liberman timebase (**LibT**), so numeric ranges are **broadly comparable** across cohorts, with small residual differences from **resampling interpolation** and **landmark detection method** (in-house peaks/troughs vs IO latencies). Cross-cohort feature comparisons in the EDA should be interpreted with this in mind, but are **not precluded**.

## 3. Grain comparison (|Δρ| ≥ 0.10)
- **PeakILateCurvature** (B, lower): ρ_long=-0.246, ρ_af=-0.482, Δρ=0.237
- **PeakIEarlyCurvature** (B, higher): ρ_long=-0.315, ρ_af=-0.545, Δρ=0.230
- **slope** (A, higher): ρ_long=-0.126, ρ_af=-0.354, Δρ=0.228
- **PeakIEarlyCurvature** (B, lower): ρ_long=-0.209, ρ_af=-0.432, Δρ=0.223
- **slope** (B, lower): ρ_long=-0.377, ρ_af=-0.592, Δρ=0.216
- **total_variance** (A, higher): ρ_long=0.125, ρ_af=0.336, Δρ=-0.211
- **amplitude** (A, higher): ρ_long=0.132, ρ_af=0.342, Δρ=-0.210
- **PeakIEarlyCurvature** (B, all): ρ_long=-0.251, ρ_af=-0.456, Δρ=0.206
- **slope** (A, all): ρ_long=-0.091, ρ_af=-0.291, Δρ=0.199
- **PeakIEarlyCurvature** (A, higher): ρ_long=-0.059, ρ_af=-0.258, Δρ=0.199
- **PeakILateCurvature** (A, higher): ρ_long=-0.101, ρ_af=-0.298, Δρ=0.197
- **amplitude** (B, lower): ρ_long=0.396, ρ_af=0.588, Δρ=-0.192
- **total_variance** (A, all): ρ_long=0.075, ρ_af=0.266, Δρ=-0.191
- **amplitude** (A, all): ρ_long=0.085, ρ_af=0.272, Δρ=-0.186
- **total_variance** (B, lower): ρ_long=0.391, ρ_af=0.569, Δρ=-0.177

## 4. Strong associations (prefer animal_freq for SHAP prior)
### animal_freq
- amplitude (A, higher): ρ=0.342, p=2.86e-10, n=323
- slope (A, higher): ρ=-0.354, p=6.044e-11, n=323
- total_variance (A, higher): ρ=0.336, p=5.938e-10, n=323
- PeakIEarlyCurvature (A, higher): ρ=-0.258, p=2.647e-06, n=323
- PeakILateCurvature (A, higher): ρ=-0.298, p=4.819e-08, n=323
- amplitude (B, lower): ρ=0.588, p=6.699e-20, n=199
- slope (B, lower): ρ=-0.592, p=3.058e-20, n=199
- total_variance (B, lower): ρ=0.569, p=1.954e-18, n=199
- PeakIEarlyCurvature (B, lower): ρ=-0.432, p=1.869e-10, n=199
- PeakILateCurvature (B, lower): ρ=-0.482, p=5.401e-13, n=199
- TroughIEarlyCurvature (B, lower): ρ=-0.329, p=2.111e-06, n=199
- TroughILateCurvature (B, lower): ρ=-0.327, p=2.491e-06, n=199
### long
- amplitude (B, lower): ρ=0.396, p=1.584e-77, n=2038
- slope (B, lower): ρ=-0.377, p=9.345e-70, n=2038
- total_variance (B, lower): ρ=0.391, p=1.429e-75, n=2038
- amplitude (B, higher): ρ=0.486, p=8.985e-91, n=1516
- slope (B, higher): ρ=-0.465, p=3.144e-82, n=1516
- total_variance (B, higher): ρ=0.484, p=7.149e-90, n=1516
- PeakIEarlyCurvature (B, higher): ρ=-0.315, p=3.3e-36, n=1516
- PeakILateCurvature (B, higher): ρ=-0.400, p=1.85e-59, n=1516
- TroughIEarlyCurvature (B, higher): ρ=-0.355, p=3.418e-46, n=1516
- TroughILateCurvature (B, higher): ρ=-0.345, p=1.065e-43, n=1516

## 5. Log-transform candidates (|skew|>1, LONG_LOG)
- total_variance (A): skew=4.75
- PeakIEarlyCurvature (A): skew=1.64
- PeakICentralCurvature (A): skew=1.61
- PeakILateCurvature (A): skew=1.91
- TroughIEarlyCurvature (A): skew=1.90
- TroughICentralCurvature (A): skew=1.54
- TroughILateCurvature (A): skew=1.87
- total_variance (B): skew=3.50
- PeakIEarlyCurvature (B): skew=1.45
- PeakICentralCurvature (B): skew=1.48
- PeakILateCurvature (B): skew=3.76
- TroughIEarlyCurvature (B): skew=1.89
- TroughICentralCurvature (B): skew=1.52
- TroughILateCurvature (B): skew=3.80

## 6. Nonlinear / scaling candidates
- PeakICentralCurvature (B, animal_freq): |ρ_s−ρ_p|=0.150
- PeakILateCurvature (B, animal_freq): |ρ_s−ρ_p|=0.200
- TroughILateCurvature (B, animal_freq): |ρ_s−ρ_p|=0.202

## 7. Data notes
- Cohort A: n_long=12187, n_animal_freq=616, dedup at_80=616, fallback=0
- Cohort B: n_long=3554, n_animal_freq=360, dedup at_80=351, fallback=9
- Includes train, validate, and test animals (exploratory).
- Scatter grids are tall (EDA layout); not print-ready without resizing.
